# QRT Directional Forecasting Challenge

This notebook summarizes the challenge setup, the financial objective, the data structure, the benchmark workflow, and the modeling choices for the QRT asset allocation performance forecasting problem.

Official challenge page: https://challengedata.ens.fr/challenges/167

## 1. Challenge Context

The challenge is about forecasting the short-term performance direction of systematic asset allocations.

Each day, a quantitative trading process may generate many candidate allocations. An allocation is a portfolio construction rule that produces a vector of asset weights. Some allocations are expected to perform well during the next trading session, while others may underperform.

The trading question is therefore simple but difficult:

> Should we trust this allocation tomorrow, or should we short it?

In machine-learning terms, this becomes a directional forecasting problem. We do not need to predict the exact next-day return. We only need to predict its sign.

## 2. Asset Allocation Definition

For a trading day $t$, an allocation $S$, and a universe of $M$ tradable instruments, let

$$
w_{S,t} = \left(w_{S,t,1}, w_{S,t,2}, \ldots, w_{S,t,M}\right)
$$

be the vector of portfolio weights of allocation $S$ at time $t$.

The allocation is normalized such that

$$
\sum_{i=1}^{M}\left|w_{S,t,i}\right| = 1.
$$

Weights may be positive or negative:

- positive weights represent long exposure,
- negative weights represent short exposure.

If $r_{i,t+1}$ is the return of asset $i$ from $t$ to $t+1$, then the realized next-day return of allocation $S$ is

$$
r_{S,t+1} = \sum_{i=1}^{M} w_{S,t,i} r_{i,t+1}.
$$

This return is the target quantity, but only its sign is evaluated.

## 3. Prediction Objective

Each observation corresponds to one pair:

$$
(t, S)
$$

where $t$ is an anonymized date and $S$ is an anonymized allocation.

For each row, we observe historical features describing the allocation over the previous 20 trading days. The goal is to predict whether the next-day return is positive or negative:

$$
y_{S,t} = \mathbf{1}\left(r_{S,t+1} > 0\right).
$$

The required prediction is binary:

$$
\widehat{y}_{S,t} \in \{0,1\}.
$$

Interpretation:

- $1$: predicted positive next-day return, trust or go long the allocation,
- $0$: predicted negative next-day return, short or avoid the allocation.

## 4. Evaluation Metric

The challenge is evaluated using accuracy:

$$
\operatorname{Accuracy}
= \frac{1}{N}\sum_{i=1}^{N}\mathbf{1}\left(\widehat{y}_i = y_i\right).
$$

Equivalently, using the return notation:

$$
\operatorname{Accuracy}
= \frac{1}{N}\sum_{i=1}^{N}
\mathbf{1}\left(\operatorname{sign}(\widehat{r}_i)=\operatorname{sign}(r_i)\right),
$$

where

$$
\operatorname{sign}(x)=
\begin{cases}
1, & x>0,\\
0, & x\leq 0.
\end{cases}
$$

Only direction matters. A model that predicts return magnitude well but gets the sign wrong will score poorly.

## 5. Data Files

The dataset is provided through four main files:

- `X_train.csv`: training features,
- `y_train.csv`: training target returns,
- `X_test.csv`: test features,
- `sample_submission.csv`: expected submission format.

All files are indexed by `ROW_ID`, a unique identifier mapping each row to a `(date, allocation)` pair.

In this local project, the files are stored in:

```text
QRT_dir_forecasting/data/raw/
```

Generated predictions are stored in:

```text
QRT_dir_forecasting/data/processed/
```

## 6. Dataset Dimensions

From the uploaded local files:

```text
X_train: 527,073 rows, 45 columns
y_train: 527,073 rows, 2 columns
X_test:   31,870 rows, 45 columns
```

The training set contains 2,522 anonymized dates and 278 unique allocations.

The test set contains 120 anonymized dates and the same 278 allocation identifiers.

The target is close to balanced:

```text
positive target rate ≈ 50.72%
```

This means that the forecasting edge is expected to be small. A naive always-positive classifier is already close to 50.7% accuracy.

## 7. Raw Feature Schema

Each row contains the following feature groups.

### Identifiers

- `TS`: anonymized timestamp.
- `ALLOCATION`: anonymized allocation name.
- `GROUP`: anonymized allocation group.

The challenge page notes that dates are anonymized and shuffled, so labels such as `DATE_0001`, `DATE_0002`, etc. should not be assumed to be continuous calendar dates.

### Return Features

- `RET_1`, `RET_2`, ..., `RET_20`

These describe the allocation's historical returns over the previous 20 trading days.

### Signed Volume Features

- `SIGNED_VOLUME_1`, `SIGNED_VOLUME_2`, ..., `SIGNED_VOLUME_20`

These describe historical signed volume or liquidity behavior of the allocation.

### Turnover Feature

- `MEDIAN_DAILY_TURNOVER`

This is a proxy for how much the allocation rebalances over time.

## 8. Signed Volume and Turnover

The signed volume of allocation $S$ at time $t$ is defined as

$$
V_{S,t} = \sum_{i=1}^{M} w_{S,t,i}V_{i,t},
$$

where $V_{i,t}$ is the total traded volume of asset $i$ during the trading session.

The turnover of an allocation measures how much its weights change from one day to the next:

$$
TO_{S,t} = \sum_{i=1}^{M}\left|w_{S,t,i} - w_{S,t-1,i}\right|.
$$

The median daily turnover is computed over a rolling 20-day window:

$$
MDT_{S,t} = \operatorname{median}\left(TO_{S,t}, TO_{S,t-1}, \ldots, TO_{S,t-20}\right).
$$

Quant intuition:

- high turnover can indicate unstable allocation rules or rapidly changing signals,
- signed volume can proxy liquidity pressure and positioning behavior,
- recent returns can capture short-term momentum, mean reversion, or decay.

## 9. Benchmark Feature Engineering

The benchmark notebook starts from base features:

```python
RET_1 ... RET_20
SIGNED_VOLUME_1 ... SIGNED_VOLUME_20
MEDIAN_DAILY_TURNOVER
```

Then it creates rolling-style summary features across the 20-day history.

### Average Historical Performance

For windows $k \in \{3,5,10,15,20\}$:

$$
\operatorname{AVG\_RET}_{k}
= \frac{1}{k}\sum_{j=1}^{k} RET_j.
$$

These features summarize recent allocation performance over different horizons.

### Cross-Sectional Average Performance

For each timestamp `TS`, the benchmark computes the average of these performance summaries across all allocations available on that date:

$$
\operatorname{CS\_AVG\_RET}_{k,t}
= \frac{1}{N_t}\sum_{S}\operatorname{AVG\_RET}_{k,S,t}.
$$

This gives market-wide or date-level context.

## 10. Volatility Features

The benchmark also creates a 20-day historical volatility proxy:

$$
\operatorname{STD\_RET}_{20}
= \operatorname{std}\left(RET_1,RET_2,\ldots,RET_{20}\right).
$$

Then it computes a cross-sectional average by timestamp:

$$
\operatorname{CS\_STD\_RET}_{20,t}
= \frac{1}{N_t}\sum_S \operatorname{STD\_RET}_{20,S,t}.
$$

Quant intuition:

- high allocation volatility may indicate unstable or risky signals,
- date-level volatility may indicate market regime,
- combining own-history and cross-sectional context can help distinguish allocation-specific effects from global conditions.

## 11. Benchmark Models

The benchmark notebook trains two models.

### Ridge Regression

A ridge model predicts the continuous next-day return:

$$
\widehat{r}_{S,t+1} = x_{S,t}^{\top}\beta.
$$

The ridge estimator solves

$$
\widehat{\beta}
= \arg\min_{\beta}\sum_i \left(r_i - x_i^{\top}\beta\right)^2 + \lambda\lVert\beta\rVert_2^2.
$$

The predicted sign is then submitted:

```python
(preds_ridge > 0).astype(int)
```

### LightGBM Regression

The benchmark also trains a LightGBM model with mean-squared-error objective:

```python
objective = "mse"
metric = "mse"
learning_rate = 1e-2
max_depth = 3
num_boost_round = 500
```

Even though the final task is classification, the model predicts a continuous return and converts it to a binary sign.

## 12. Cross-Validation Strategy

The benchmark performs cross-validation by splitting unique timestamps `TS`, not individual rows.

This matters because rows from the same timestamp are cross-sectionally related. If we randomly split individual rows, the validation set could contain allocations from the same date as the training set, causing optimistic leakage.

The benchmark approach is:

1. Extract unique training dates.
2. Split dates using K-fold cross-validation.
3. Train on rows whose `TS` belongs to training dates.
4. Validate on rows whose `TS` belongs to validation dates.

This is closer to the real problem: generalizing to unseen dates.

## 13. Submission Format

The submission file must be indexed by `ROW_ID` and contain a `target` column.

The benchmark creates binary predictions as follows:

```python
(preds_lgbm > 0).astype(int).to_csv("preds_lgbm_bench.csv")
```

So each prediction should be:

```text
0 or 1
```

not a continuous return.

## 14. Baseline Difficulty

The target is almost balanced:

$$
\mathbb{P}(r_{S,t+1}>0) \approx 50.72\%.
$$

The public benchmark score reported on the challenge page is approximately:

```text
0.5079
```

This tells us that the signal-to-noise ratio is very low. The project should therefore focus on robust validation and small, stable improvements rather than expecting large accuracy gains.

## 15. Modeling Mindset

This challenge should be treated as a noisy cross-sectional time-series forecasting problem.

Important principles:

- validate by timestamp, not by random rows,
- avoid leakage from test or future dates,
- compare against simple baselines,
- track both accuracy and class balance,
- inspect feature importance for economic plausibility,
- be skeptical of small improvements unless they are stable across folds,
- remember that a high-quality trading signal may only improve accuracy by a small amount.

The clean project notebook should therefore be built as a controlled research workflow: data audit, feature engineering, validation design, baseline models, diagnostics, and final submission generation.